In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
# !pip install "tmd==2.3.0"

In [ ]:
# !pip install torch_geometric

In [ ]:
from morphoclass.data import MorphologyDataset
from morphoclass.data.loader import build_dataloader

In [ ]:
import torch
import torch_geometric
import torch_scatter
from torch_geometric.data import Data

print(torch.__version__, torch.version.cuda)
print(torch_geometric.__version__)
print(torch_scatter.__version__)


In [ ]:
from morphoclass.models.coriander_net import CorianderNet
help(CorianderNet)

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)


In [ ]:
import tmd
print(tmd.__version__)

In [ ]:
# !jupyter kernelspec list

In [ ]:
# !jupyter kernelspec inspect morphoclass


In [ ]:
# !cat ~/.local/share/jupyter/kernels/morphoclass/kernel.json


In [ ]:
# !ls /gscratch/scrubbed/emazuh/morphoclass

In [ ]:
# !jupyter kernelspec remove morphoclass -y

In [ ]:
# import sys
# print(sys.executable)


In [ ]:
# !pip list

In [ ]:
# !python -m ipykernel install --user --name morphoclass --display-name "Python (morphoclass)"

In [ ]:
# !conda env list

In [ ]:
dataset = MorphologyDataset(
    root_dir="data",
    categories=["interneuron_A"],  # single class for demo
    transforms=None
)

In [ ]:
dataloader = build_dataloader(dataset, batch_size=1, shuffle=False)

sample, label = next(iter(dataloader))
print("Sample shape:", sample.x.shape if hasattr(sample, 'x') else "single neuron object")
print("Label:", label)

# ========================================================
# Step 3: Define a minimal PersLay model
# ========================================================

from morphoclass.models.perslay import PersLay
import torch

# Minimal PersLay config
model = PersLay(
    in_features=dataset.num_node_features,
    hidden_features=16,
    n_classes=1  # single class for demo
)

# Send model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
sample = sample.to(device)

# ========================================================
# Step 4: Quick forward pass (mock “train” not needed)
# ========================================================

model.eval()
with torch.no_grad():
    output = model(sample)
print("Raw output:", output)

# ========================================================
# Step 5: Explainability using Captum (Integrated Gradients)
# ========================================================

from captum.attr import IntegratedGradients

ig = IntegratedGradients(model)

# For demonstration: attribute the output w.r.t input features
attributions = ig.attribute(sample.x, target=0)  # target=0 for single-class output
print("Attributions shape:", attributions.shape)
print("Attributions (first 10 nodes):", attributions[:10])

# ========================================================
# Step 6: Optional: Map back to neuron tree (simple visualization)
# ========================================================

import matplotlib.pyplot as plt
import numpy as np

# Map attribution magnitude per node
node_scores = attributions.abs().sum(dim=1).cpu().numpy()

plt.figure(figsize=(8,4))
plt.bar(np.arange(len(node_scores)), node_scores)
plt.xlabel("Node index")
plt.ylabel("Attribution magnitude")
plt.title("PersLay + Integrated Gradients: Node Importance")
plt.show()
